# Notebook 2 : Mécanismes ponctuels

In [3]:
%reload_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().parent                 
if str(ROOT) not in sys.path:                     
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, SelectionSlider, Dropdown

import online_dp as dp
from online_dp.config import Config
from online_dp.cache import compute_or_load
from online_dp import data, metrics, basis, gam, mechanisms, vfast, viz

cfg = Config(
    data_dir=str(ROOT.parent / "data" / "DataDiffusionDeepCourboGen") + "/",   
    cache_dir=str(ROOT / "cache"),
    n_panel=500, n_target=50, seed=0,                
)
dp.cache.CACHE = pathlib.Path(cfg.cache_dir)       # cache partagé par les 4 notebooks
cfg

Config(data_dir='/home/G70186/data/DataDiffusionDeepCourboGen/', cache_dir='/home/G70186/onlinedp_stage26/cache', N=1000, n_panel=500, n_target=50, seed=0, calendar_start='2022-10-02 20:00:00', slots_per_day=48, pmax=47, n_groups=100, fit_subsample=10000, nmf_max_iter=500, nmf_tol=0.0001, clip_quantile=0.95, delta_dp=1e-05)

In [4]:
D = compute_or_load(cfg.key('dataset'), lambda: data.build_dataset(cfg))

[cache] 'dataset__N1000_np500_nt50_s0' rechargé (joblib).


In [5]:
df_agg = D['df_agg']
L_true = df_agg.values
T, S   = L_true.shape
dates_str = [d.strftime('%Y-%m-%d') for d in df_agg.index]

## Sensibilité ponctuelle naive


In [6]:
P_max_kVA, dt_h = 12.0, 0.5
sens   = (P_max_kVA * dt_h) / cfg.n_target
DELTA  = cfg.delta_dp
EPS_GRID = np.arange(1, 101, 5)
METRICS  = list(metrics.METRIC_LABEL)  

## Mécanismes ponctuels Laplacien et Gaussien, budget quotidien $\varepsilon$ et $\frac{\varepsilon}{T}$

In [7]:
def _compute_pointwise_metrics():
    out = {}
    for mode in ['per_day', 'sequential']:
        lap = mechanisms.lpa_pointwise_grid(L_true, EPS_GRID, sens, mode=mode)
        gau = mechanisms.gaussian_pointwise_grid(L_true, EPS_GRID, sens, DELTA, mode=mode)
        for name, tens in [('laplace', lap), ('gaussian', gau)]:
            md = metrics.compute_wpa_metrics(tens, L_true)              # metric -> (n_eps, T)
            out[(name, mode)] = {m: metrics.stats_over_axis(v, axis=1, q_lo=0.05, q_hi=0.95)
                                 for m, v in md.items()}                # dict mean/lo/hi, chacun (n_eps,)
    return out

# results = compute_or_load(cfg.key('pointwise_metrics'), _compute_pointwise_metrics)   # <- decommente pour cacher
results = _compute_pointwise_metrics()

In [8]:
slots = np.arange(S)

@interact(eps=IntSlider(min=1, max=100, step=5, value=10, description='eps',
                        continuous_update=False, layout=dict(width='430px')),
          day=IntSlider(min=0, max=T - 1, step=1, value=0, description='jour',
                        continuous_update=False, layout=dict(width='430px')))
def show_pointwise(eps, day):
    Ld = L_true[day]
    rs = int(eps) * 100_000 + int(day)        # graine variable
    pairs = {
        1: (mechanisms.lpa_pointwise(Ld[None, :], eps, sens, 'per_day', seed=rs)[0],
            mechanisms.gaussian_pointwise(Ld[None, :], eps, sens, DELTA, 'per_day', seed=rs + 1)[0]),
        2: (mechanisms.lpa_pointwise(Ld[None, :], eps, sens, 'sequential', T_compose=T, seed=rs)[0],
            mechanisms.gaussian_pointwise(Ld[None, :], eps, sens, DELTA, 'sequential', T_compose=T, seed=rs + 1)[0]),
    }
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
        subplot_titles=(f"budget eps={eps} par jour",
                        f"budget eps/T={eps/T:.3f} par jour"))
    for col, (lap, gau) in pairs.items():
        sl = (col == 1)
        fig.add_trace(go.Scatter(x=slots, y=Ld, mode='lines', name='réel', legendgroup='r',
                      showlegend=sl, line=dict(color=viz.COLOR_REAL, width=2.5)), row=1, col=col)
        fig.add_trace(go.Scatter(x=slots, y=lap, mode='lines', name='Laplace', legendgroup='l',
                      showlegend=sl, line=dict(color=viz.MECH_COLOR['laplace'], width=1.6)), row=1, col=col)
        fig.add_trace(go.Scatter(x=slots, y=gau, mode='lines', name='Gaussien', legendgroup='g',
                      showlegend=sl, line=dict(color=viz.MECH_COLOR['gaussian'], width=1.6)), row=1, col=col)
    fig.update_xaxes(title_text='creneau')
    fig.update_yaxes(title_text='conso (kVA)', row=1, col=1)
    fig.update_layout(height=440, width=1060, template='plotly_white',
        title=dict(text=f"L(t) vs Perturbation ponctuelle - jour {dates_str[day]}", x=0.5,
                   font=dict(size=15, color=viz.COLOR_REAL)),
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
        margin=dict(t=85, b=55))
    fig.show()

interactive(children=(IntSlider(value=10, continuous_update=False, description='eps', layout=Layout(width='430…

## Compromis utilité / confidentialité 

In [9]:
@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='320px'), style={'description_width': '80px'}))
def show_perf(metric):
    fig = go.Figure()
    for mech in ['laplace', 'gaussian']:
        for mode in ['sequential', 'per_day']:
            s = results[(mech, mode)][metric]
            color = viz.MECH_COLOR[mech]
            fig.add_trace(viz.band(EPS_GRID, s['hi'], s['lo'], fillcolor=viz.rgba(color, 0.12)))
            fig.add_trace(go.Scatter(
                x=EPS_GRID, y=s['mean'], mode='lines',
                name=f"{mech} - {viz.MODE_LABEL[mode]}",
                line=dict(color=color, width=2.2, dash=viz.MODE_DASH[mode])))
    fig.update_layout(
        title=dict(text=f"Utilité des mécanismes ponctuels vs eps - {metrics.METRIC_LABEL[metric]}", x=0.5,
                   font=dict(size=15, color=viz.COLOR_REAL)),
        xaxis_title='eps', yaxis_title=metrics.METRIC_LABEL[metric],
        height=480, width=840, template='plotly_white', hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
    fig.update_yaxes(type='log')      # 'linear' si tu preferes
    fig.show()

interactive(children=(Dropdown(description='metrique', index=1, layout=Layout(width='320px'), options=(('MAPE …

## Effet du clipping pour le calcul de la sensibilité sur l'utilité des deux mécanismes ponctuels (Laplace et Gaussien)

In [10]:
# Seuils de clipping C (sur le PANEL public) + agregats cibles clippés a C (introduit un biais)

df_daily = D['df_daily']
target_users = D['target_users']
panel_tensor = D['panel_tensor']
target_tensor, _ = data.household_tensor(df_daily, target_users)     # (n_target, T, 48)
panel_vals = panel_tensor.reshape(-1)                                # charges individuelles du panel (public)

P_max_kVA, dt_h = 12.0, 0.5
CLIP = {
    'naif': P_max_kVA * dt_h,                        # borne theorique (puissance souscrite max)
    'q90' : float(np.quantile(panel_vals, 0.90)),
    'q95' : float(np.quantile(panel_vals, 0.95)),
    'q99' : float(np.quantile(panel_vals, 0.99)),
    'max' : float(panel_vals.max()),                 # max empirique du panel
}
METHODS = ['naif', 'q90', 'q95', 'q99', 'max']

clip_setup = {}
for meth in METHODS:
    C = CLIP[meth]
    L_clipped = np.clip(target_tensor, 0.0, C).mean(axis=0)          # (T, 48), biaise si C < pics
    clip_setup[meth] = (C / cfg.n_target, L_clipped)                 # (sensibilite scalaire, agregat clippe)

print("seuils C :", {k: round(v, 3) for k, v in CLIP.items()})
print("sanity (max empirique <= borne naive) :", CLIP['max'] <= CLIP['naif'])

seuils C : {'naif': 6.0, 'q90': 1.645, 'q95': 2.406, 'q99': 4.403, 'max': 26.216}
sanity (max empirique <= borne naive) : False


In [11]:
@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in metrics.METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='320px'), style={'description_width': '80px'}),
          eps=IntSlider(min=1, max=100, step=5, value=10, description='eps',
                        continuous_update=False, layout=dict(width='430px')))
def show_clipping(metric, eps):
    fig = go.Figure()
    for mech in ['laplace', 'gaussian']:
        xs, ys = [], []
        for j, meth in enumerate(METHODS):
            sens_m, L_clip = clip_setup[meth]
            seed = int(eps) + j + (0 if mech == 'laplace' else 500)
            if mech == 'laplace':
                noisy = mechanisms.lpa_pointwise(L_clip, eps, sens_m, 'per_day', seed=seed)
            else:
                noisy = mechanisms.gaussian_pointwise(L_clip, eps, sens_m, DELTA, 'per_day', seed=seed)
            vals = metrics.per_day_metrics(L_true, noisy)[metric]     # (T,) compare a l'agregat VRAI
            xs += [meth] * len(vals); ys += list(vals)
        fig.add_trace(go.Box(x=xs, y=ys, name=mech, marker_color=viz.MECH_COLOR[mech],
                             boxpoints=False, line=dict(width=1.3)))
    fig.update_layout(
        boxmode='group',
        title=dict(text=f"Utilité vs méthode de clipping  (budget eps/jour, eps={eps})  -  {metrics.METRIC_LABEL[metric]}",
                   x=0.5, font=dict(size=14, color=viz.COLOR_REAL)),
        xaxis_title='méthode de clipping (calcul de la sensibilite)',
        yaxis_title=metrics.METRIC_LABEL[metric],
        height=470, width=860, template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
    # fig.update_yaxes(type='log')    # decommente si 'naif' ecrase l'echelle
    fig.show()

interactive(children=(Dropdown(description='metrique', index=1, layout=Layout(width='320px'), options=(('MAPE …